In [2]:
import argparse
import glob
import numpy as np
import pandas as pd
import tables as tb
from   typing      import Callable
from   typing      import Optional
from   typing      import List
from   pandas      import DataFrame
from   pandas      import Series

from sklearn.neighbors import NearestNeighbors

from invisible_cities.core.exceptions     import TimeEvolutionTableMissing
from invisible_cities.reco.corrections import read_maps, apply_all_correction
from invisible_cities.types.symbols import NormStrategy

def get_args():
    parser = argparse.ArgumentParser(
        description="Process a NEXT100 run and build summaries")
    parser.add_argument(
        "-r", "-run", "--run",
        dest="run_number",
        type=int,
        required=True,
        help="Run number to analyse, e.g. 15107")
    parser.add_argument(
        "-m", "--map",
        dest="map_name",
        type=str,
        default='map_MC_4bar_15063.h5',
        help="Name of the correction map")
    parser.add_argument(
        "-s", "--save",
        dest="save_name",
        type=str,
        default="run_summary.h5",
        help="Name of the file to save")
    return parser.parse_args()

# args        = get_args()
run_number  = 15568 #args.run_number 

source_path = '/mnt/netapp1/Store_next_data/NEXT100/data/{run_n}/hdf5/prod/*/*/sophronia/trigger2/'.format(run_n = run_number)
data_path = source_path + '*/*'


store_path  = '/mnt/lustre/scratch/nlsas/home/usc/ie/mpm/NEXT100/data/new_HE_runs/' 
save_path_summary = store_path + '/{run_n}/'.format(run_n = run_number) + '' #args.save_name 
map_path = store_path + 'map_MC_4bar_15063.h5' #args.map_name

def drop_isolated_clusters(distance: List[float] = [10., 10.], nhit: int = 3,
                           variables: List[str] = []) -> Callable:
    dist = np.sqrt(distance[0] ** 2 + distance[1] ** 2)

    def drop(df: pd.DataFrame) -> pd.DataFrame:
        if len(df) == 0:
            return df

        xy = df[['X', 'Y']].values

        try:
            nbrs = NearestNeighbors(radius=dist, algorithm='ball_tree').fit(xy)
            neighbors = nbrs.radius_neighbors(xy, return_distance=False)
            mask = np.array([len(neigh) > nhit for neigh in neighbors])
        except Exception as e:
            print(f"Error in NearestNeighbors: {{e}}")
            return df.iloc[:0]  # fallback: return empty

        pass_df = df.loc[mask].copy()

        if not pass_df.empty and variables:
            with np.errstate(divide='ignore', invalid='ignore'):
                columns = pass_df.loc[:, variables]
                scale = df[variables].sum().values / columns.sum().values
                columns *= scale
                pass_df.loc[:, variables] = columns

        return pass_df

    return drop

def get_fname_info(path):
    file = path.split('/')[-1]
    split_name = file.split('_')
    run_n, file_n, ldc_n = split_name[1], split_name[2], split_name[3]
    return run_n, file_n, ldc_n

def hits_summary(group, fname, coords = ['X', 'Y', 'Z'], ener = 'Ec'):
    def dcoord(group, coords, i):
        c = coords[i]
        return group[c].max() - group[c].min()
    def barycenter(group, coords, i):
        return (group[coords[i]] * group['Q']).sum() / group['Q'].sum() #changed to Q because makes more sense
    
    run_n, file_n, ldc_n = get_fname_info(fname)
    
    return pd.Series({
        'run_n': str(run_n),
        'file_n': str(file_n),
        'ldc_n': str(ldc_n),
        'time': group['time'].unique()[0],
        'dX': dcoord(group, coords, 0),
        'dY': dcoord(group, coords, 1),
        'dZ': dcoord(group, coords, 2),
        'X_min': group[coords[0]].min(),
        'Y_min': group[coords[1]].min(),
        'Z_min': group[coords[2]].min(), 
        'Rmax': np.sqrt(group['X']**2 + group['Y']**2).max(),
        # 'X_max': group[coords[0]].max(),
        # 'Y_max': group[coords[1]].max(),
        # 'Z_max': group[coords[2]].max(),
        'X_bary': barycenter(group, coords, 0),
        'Y_bary': barycenter(group, coords, 1),
        'Z_bary': barycenter(group, coords, 2),
        'total_E': group['E'].sum(),
        # 'total_E_noneg': group['E'].clip(lower=0).sum(), #do the sum but with no negative hits
        'total_energy': group[ener].sum(),
        # 'total_energy_noneg': group[ener].clip(lower=0).sum(), #do the sum but with no negative hits
        'num_hits': int(len(group)),
        # 'num_neg_hits': (group['E'] < 0).sum()
    })

def create_hits_summary(reco, corr_fun, dropper, fname):
    # Correct energy
    factor = corr_fun(reco.X, reco.Y, reco.Z, reco.time)
    reco['Ec'] = reco.E * factor
    #correct energy in the borders (for NaN hits)
    factor_border = get_coef([479],[0],[0],[1])
    reco["Ec_border"] = reco.E * factor_border
    # add the latter energy
    reco["Ec"] = reco["Ec"].fillna(reco["Ec_border"])
    reco = reco.groupby(['event', 'npeak'], group_keys=False).apply(dropper)
    return reco.groupby(['event', 'npeak']).apply(lambda group: hits_summary(group, fname)).reset_index()


files = sorted(glob.glob(data_path + '*'), key=lambda x: (x.split('/')[-2], int(x.split('/')[-1].split('_')[2])))

# Energy correction
maps = read_maps(map_path)

get_coef  = apply_all_correction(maps
                                 , apply_temp = False
                                 , norm_strat = NormStrategy.kr)
# Cluster dropping
dropper = drop_isolated_clusters(distance = [15.,15.], nhit = 3, variables = ['Ec'])

# time_to_Z = get_df_to_z_converter(maps) if maps.t_evol is not None else identity

# for i, f in enumerate(files):
#     try:
#         dst = pd.read_hdf(f, 'DST/Events')
#         reco = pd.read_hdf(f, 'RECO/Events')
#     except Exception as e:
#         print(f"Skipping corrupted/invalid file: {f}")
#         print(f"Error: {e}")
#         continue

#     if reco.empty:
#         print(f"Skipping empty file: {f}")
#         continue

#     # create summary
#     reco_summary = create_hits_summary(reco, get_coef, dropper, f)

#     dst.to_hdf (save_path_summary, key = 'DST/Events', mode = 'a', append= True)
#     reco_summary.to_hdf(save_path_summary, key = 'RECO/Events_summary', mode = 'a', append= True)
#     print(i)

In [1]:
import argparse
import glob
import numpy as np
import pandas as pd
import tables as tb
from   typing      import Callable
from   typing      import Optional
from   typing      import List
from   pandas      import DataFrame
from   pandas      import Series

from sklearn.neighbors import NearestNeighbors

from invisible_cities.core.exceptions     import TimeEvolutionTableMissing
from invisible_cities.reco.corrections import read_maps, apply_all_correction
from invisible_cities.types.symbols import NormStrategy

In [2]:
run_number  = 15604 #args.run_number 

source_path = '/mnt/netapp1/Store_next_data/NEXT100/data/{run_n}/hdf5/prod/*/*/sophronia/trigger2/'.format(run_n = run_number)
data_path = source_path + '*/*'

files = sorted(glob.glob(data_path + '*'), key=lambda x: (x.split('/')[-2], int(x.split('/')[-1].split('_')[2])))


In [3]:
f = files[0]


dst = pd.read_hdf(f, 'DST/Events')
reco = pd.read_hdf(f, 'RECO/Events')

# factor= get_coef(reco.X, reco.Y, reco.Z, reco.time)
# factor_border = get_coef([479],[0],[0],[1])

# reco['Ec'] = reco.E * factor
# reco["Ec_border"] = reco.E * factor_border

# reco["Ec"] = reco["Ec"].fillna(reco["Ec_border"])

# dropper = drop_isolated_clusters(distance = [15.,15.], nhit = 3, variables = ['Ec'])



In [4]:
sipm_threshold = 7

In [77]:
gr_cols = ['event', 'npeak', 'Z']
hits = reco.copy()
energy_sum = hits.groupby(gr_cols)['E'].sum().rename('E_sli')
hits = hits.merge(energy_sum, on=gr_cols, how='left')


hits_n = hits[hits.Q > sipm_threshold].copy()
charge_sum = hits_n.groupby(gr_cols)['Q'].sum().rename('Q_sli')
hits_n = hits_n.merge(charge_sum, on=gr_cols, how='left')

hits_n['Q_norm'] = hits_n['Q'] / hits_n['Q_sli']
hits_n['E_norm'] = hits_n['E_sli'] * hits_n['Q_norm']

e0sum   = hits  .groupby('event')['E'].sum().values
e1sum   = hits_n.groupby('event')['E_norm'].sum().values
h1nhits = hits_n.groupby('event')['E_norm'].count().values
factor  = np.repeat(e0sum/e1sum, h1nhits)
hits_n['E_norm'] = factor * hits_n['E_norm']

In [78]:
hits_n.groupby(['event', 'npeak', 'Z'])['E_norm'].sum().reset_index().merge(energy_sum, on = ['event', 'npeak', 'Z'])

,event,npeak,Z,E_norm,E_sli
0,29,32,553.318250,-1.740437,-1.691314
1,29,32,556.295000,-6.136993,-5.963780
2,29,32,564.796625,1.614237,1.568676
3,29,32,580.352125,2.580445,2.507614
4,29,32,620.592500,1349.050488,1310.974461
...,...,...,...,...,...
1766,722,24,365.988000,16.748832,16.685522
1767,722,24,370.002000,15.483254,15.424727
1768,722,24,373.932000,13.572870,13.521564
1769,722,24,385.502750,13.637299,13.585750


In [79]:
hits_n.groupby('event').E_norm.sum()

event
29      90443.632106
50     179942.470983
71     142676.095920
106     79518.850020
141     56797.882189
162    145777.268184
169     69583.556361
211     77104.815358
225    382554.120316
260    124804.385849
274    132838.719784
309    330199.349513
330    428200.842240
344     80744.520404
393    290405.786054
400    421284.766790
435    138633.618225
442    178145.547634
456    245040.600269
484    120275.691447
491    431878.775832
498    100859.684100
540    210216.224054
554     84856.645570
561    280344.970943
575    194618.668811
582    186019.434765
589     88863.381877
645    186136.085461
687    291035.827979
722    267369.807459
Name: E_norm, dtype: float64

In [80]:
hits_n.groupby('event').E.sum()

event
29      75114.453273
50     159159.938272
71     124608.461704
106     69305.944286
141     50800.494237
162    133518.599128
169     62141.216547
211     68431.743476
225    328621.921516
260    106315.651987
274    115521.957566
309    281654.337005
330    372341.610525
344     70652.851892
393    254952.148420
400    360530.247298
435    122561.386155
442    156860.899805
456    208396.568018
484    106758.087330
491    373281.782169
498     87132.690056
540    186096.658873
554     75657.578091
561    247429.935230
575    168201.930212
582    160073.910826
589     78256.030015
645    164246.358993
687    250010.068808
722    241970.219532
Name: E, dtype: float64

In [5]:
def apply_Q_cut(hits_df: pd.DataFrame, 
                sipm_threshold: float,
                preserve_event_light: bool = True):
    '''
    Redistributes the total energy of each (event, Z) group among the hits
    that survive a charge threshold, proportional to their charge.
    '''
    group_columns = ['event', 'npeak', 'Z']
    hits = hits_df.copy()
    # 1. Calcular energía total y carga total por (event, Z) ANTES del corte
    energy_sum = hits.groupby(group_columns)['E'].sum().rename('E_evz')
    hits = hits.merge(energy_sum, on=group_columns, how='left')
    # 2. Aplicar el corte
    hits_n = hits[hits.Q >= sipm_threshold].copy()
    charge_sum = hits_n.groupby(group_columns)['Q'].sum().rename('Q_evz')
    # 3. Añadir las columnas globales de suma de energía y carga
    hits_n = hits_n.merge(charge_sum, on=group_columns, how='left')
    # 4. Redistribuir proporcionalmente
    hits_n['Q_norm'] = hits_n['Q'] / hits_n['Q_evz']
    hits_n['E_norm'] = hits_n['E_evz'] * hits_n['Q_norm']

    if (preserve_event_light):
        e0sum   = hits  .groupby('event')['E'].sum().values
        e1sum   = hits_n.groupby('event')['E_norm'].sum().values
        h1nhits = hits_n.groupby('event')['E_norm'].count().values
        factor  = np.repeat(e0sum/e1sum, h1nhits)
        hits_n['E_norm'] = factor * hits_n['E_norm']
    hits_n['E']  = hits_n['E_norm'].values
    hits_n = hits_n.drop(['E_evz', 'Q_evz', 'Q_norm', 'E_norm'], axis = 1)

    return hits_n

In [31]:
cut = apply_Q_cut(reco, 7, preserve_event_light=False)

In [32]:
cut.groupby(['event', 'npeak', 'Z']).sum().E

event  npeak  Z         
29     32     553.318250      -1.691314
              556.295000      -5.963780
              564.796625       1.568676
              580.352125       2.507614
              620.592500    1310.974461
                               ...     
722    24     365.988000      16.685522
              370.002000      15.424727
              373.932000      13.521564
              385.502750      13.585750
              401.978625      10.644095
Name: E, Length: 1771, dtype: float64

In [33]:
reco.groupby(['event', 'npeak', 'Z']).sum().E

event  npeak  Z         
29     32     553.318250    -1.691314
              556.295000    -5.963780
              560.302375    -1.436703
              564.796625     1.568676
              568.232750    -3.050557
                              ...    
722    24     398.202875    11.405120
              401.978625    10.644095
              406.019250    12.568759
              409.537250    11.871096
              413.704750     4.295178
Name: E, Length: 3070, dtype: float64